## Tools in LangChain

A Tool is a Python function that an AI agent can call to perform a specific task.

Imagine you're building an AI assistant for a banking application.

A customer asks:
"What is my current account balance?"

An LLM alone cannot access the customer's live bank balance. It needs a function that connects to the bank's database or API.

That function becomes a Tool in LangChain.

Think of it this way:
- LLM = Brain — Understands the question and decides what is needed.
- Tool = Hands — Performs an action or retrieves information.
- Agent = Decision-maker — Coordinates the LLM and tools.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [4]:
from langchain_groq import ChatGroq

model=ChatGroq(
    model="qwen/qwen3.8-27b"
)

response = model.invoke("Explain about AI.")
response

AIMessage(content='**Artificial Intelligence (AI)** is the field of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, perception, and language understanding.\n\nHere’s a comprehensive breakdown of AI:\n\n### 1. Core Concept\nAI is not a single technology but a broad field encompassing many approaches. At its heart, AI is about **building machines that can mimic human cognitive functions**.\n\n- **Narrow AI (Weak AI):** AI systems designed for a specific task (e.g., chess engines, voice assistants, recommendation algorithms). This is the type of AI we have today.\n- **General AI (Strong AI):** Hypothetical AI that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence. This does not yet exist.\n\n### 2. Key Technologies\n- **Machine Learning (ML):** A subset of AI where systems learn from data r

In [5]:
from langchain.tools import tool

@tool #if we write this decorator, it will automatically register this function as a tool in the model's tool registry.
def get_weather(location:str) -> str:
    """Get the current weather for a given location."""
    # Here you would implement the logic to get the weather for the location
    return f"The current weather in {location} is sunny with a temperature of 25°C."


#after creating the tool , in order to unse in the model we need to bind it
model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'aydp6vt5w', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 280, 'total_tokens': 307, 'completion_time': 0.078313268, 'completion_tokens_details': None, 'prompt_time': 0.021882631, 'prompt_tokens_details': None, 'queue_time': 0.856400675, 'total_time': 0.100195899}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ab4f-7a79-765f-bfe7-377374a17c36-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'aydp6vt5w', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 280, 'output_tokens': 27, 'total_tokens': 307}


In [7]:
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

Tool:get_weather
Args:{'location': 'New York'}


Tool Execution Loops

In [8]:
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)


for tool_call in ai_message.tool_calls:
    print(tool_call)
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


final_response = model_with_tools.invoke(messages)
print(final_response)

{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '8wn0mtnxh', 'type': 'tool_call'}
content='The current weather in New York is sunny with a temperature of 25°C.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 341, 'total_tokens': 359, 'completion_time': 0.03798698, 'completion_tokens_details': None, 'prompt_time': 0.027558647, 'prompt_tokens_details': None, 'queue_time': 0.252455528, 'total_time': 0.065545627}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ab4f-80dd-74bf-bf55-1994a1fd24f2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 341, 'output_tokens': 18, 'total_tokens': 359}


In [9]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8wn0mtnxh', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 280, 'total_tokens': 307, 'completion_time': 0.074727056, 'completion_tokens_details': None, 'prompt_time': 0.021793643, 'prompt_tokens_details': None, 'queue_time': 0.340716562, 'total_time': 0.096520699}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1ab4f-7ebc-7009-8dfc-70c3946a9793-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '8wn0mtnxh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 27, 'total_tokens': 307}),
 ToolMessage(content='The 